# 🌾 Crop Recommendation Model Development
### Precision Agriculture Decision Support with Machine Learning

This notebook details the end-to-end machine learning pipeline for **Crop Recommendation**:
- **Exploratory Data Analysis (EDA)**
- **Nutrient and Climatic Distribution Analysis**
- **Data Preprocessing & Label Encoding**
- **Model Training: Random Forest vs XGBoost**
- **Performance Evaluation (Accuracy, F1-Score, Confusion Matrix)**
- **Feature Importance Interpretation**
- **Model Serialization**

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Locate dataset
data_path = '../datasets/Crop_recommendation.csv' if os.path.exists('../datasets/Crop_recommendation.csv') else 'datasets/Crop_recommendation.csv'
df = pd.read_csv(data_path)
print(f'Dataset Dimensions: {df.shape}')
df.head()

In [2]:
# Check Nulls, Data Types & Class Balance
print('Missing Values:\n', df.isnull().sum())
print('\nDuplicate Records:', df.duplicated().sum())
print(f'\nTotal Unique Crops: {df["label"].nunique()}')
print(sorted(df['label'].unique()))
df.describe()

In [3]:
# Feature Distributions
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
plt.figure(figsize=(14, 10))
for i, col in enumerate(features):
    plt.subplot(3, 3, i + 1)
    sns.histplot(df[col], kde=True, color='#2E7D32')
    plt.title(f'Distribution of {col}', fontsize=11)
plt.tight_layout()
plt.show()

In [4]:
# Correlation Matrix Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, cmap='YlGnBu', fmt='.2f', cbar=True)
plt.title('Soil and Environmental Feature Correlation Heatmap')
plt.show()

In [5]:
# Train-Test Split with Stratification
X = df[features].values
y_raw = df['label'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}')

In [6]:
# 1. Random Forest Classifier Training
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_preds)
print(f'Random Forest Test Accuracy: {rf_acc * 100:.2f}%')

In [7]:
# 2. XGBoost Classifier Training
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='mlogloss')
xgb.fit(X_train_scaled, y_train)
xgb_preds = xgb.predict(X_test_scaled)
xgb_acc = accuracy_score(y_test, xgb_preds)
print(f'XGBoost Test Accuracy: {xgb_acc * 100:.2f}%')

In [8]:
# Classification Report & Confusion Matrix
print(classification_report(y_test, rf_preds, target_names=le.classes_))

cm = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Random Forest — Confusion Matrix')
plt.xlabel('Predicted Crop')
plt.ylabel('Actual Crop')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [9]:
# Feature Importance
feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='crest')
plt.title('Feature Importances — Random Forest Classifier')
plt.xlabel('Gini Importance')
plt.show()

In [10]:
# Save Model & Preprocessor Bundle
models_dir = '../models' if os.path.exists('../models') else 'models'
os.makedirs(models_dir, exist_ok=True)
joblib.dump(rf, os.path.join(models_dir, 'crop_recommendation_model.pkl'))
joblib.dump({'scaler': scaler, 'label_encoder': le, 'features': features, 'classes': le.classes_.tolist()},
            os.path.join(models_dir, 'crop_recommendation_preprocessor.pkl'))
print('Crop Recommendation Model saved successfully!')